# 🎨 Novel Visualiser — ComfyUI Production Workbook
### The "6 + 2" Specialized Workflows with Anti-Hang Subprocess & Symlink Hierarchy

This workbook runs the complete **Novel Visualiser** image generation pipeline on Kaggle/Colab with GPU acceleration (P100 / T4 / L4 / A100):
- **Zero Pipe Deadlocks**: ComfyUI runs in the background logging directly to a disk stream, eliminating OS 64KB buffer hangs.
- **Dual-Path Symlinks**: Preserves nested subfolders and flat filenames so API payloads never throw missing model errors.
- **Native Regional Isolation**: SolidMask + MaskComposite + ConditioningSetMask eliminates tag bleeding between characters.
- **8 Specialized Production Workflows**: Character Foundry, Location Master Plate + Depth/Lineart, Prop Sheet, Solo Shot, Multi-Subject Shot, Vista Environment, Surgical Inpaint, and 4K AnimeSharp Master Upscale.

## 📦 Cell 1: Install ComfyUI & Required Custom Nodes
Installs ComfyUI, IPAdapter Plus, Impact Pack, and auxiliary libraries non-interactively without prompting.

In [ ]:
import os
import subprocess
import sys

WORKSPACE = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
COMFY_DIR = os.path.join(WORKSPACE, "ComfyUI")

def run_cmd(cmd, cwd=None):
    print(f"▶ Running: {cmd}")
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)

if not os.path.exists(COMFY_DIR):
    run_cmd(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}")

CUSTOM_NODES = {
    "ComfyUI_IPAdapter_plus": "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
    "ComfyUI-Impact-Pack": "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "ComfyUI-Impact-Subpack": "https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git",
}

nodes_dir = os.path.join(COMFY_DIR, "custom_nodes")
for name, url in CUSTOM_NODES.items():
    target = os.path.join(nodes_dir, name)
    if not os.path.exists(target):
        run_cmd(f"git clone --depth 1 {url} {target}")

# Install requirements silently without hanging on prompts
run_cmd(f"{sys.executable} -m pip install -q --no-input -r requirements.txt", cwd=COMFY_DIR)
run_cmd(
    f"{sys.executable} -m pip install -q --no-input "
    "ultralytics opencv-python-headless GitPython piexif segment-anything scikit-image dill matplotlib websocket-client httpx"
)

print("✅ Cell 1 Complete: ComfyUI and Custom Nodes installed.")

## 🔗 Cell 2: Mount & Link nv-comfyui-model-library Dataset
Discovers models and creates dual-path symlinks (both relative hierarchy and flat filename) into `ComfyUI/models/`.

In [ ]:
import json
from pathlib import Path

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/content"), Path(WORKSPACE)]
DATASET_ROOT = None

for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    candidates = [root / "nv-comfyui-model-library"] + list(root.glob("**/nv-comfyui-model-library"))
    for c in candidates:
        if c.exists() and (c / "checkpoints").exists():
            DATASET_ROOT = c
            break
    if DATASET_ROOT:
        break

if DATASET_ROOT is None:
    raise FileNotFoundError("Could not find nv-comfyui-model-library in /kaggle/input or /content!")

print(f"📂 Found Dataset at: {DATASET_ROOT}")

CATEGORY_MAP = {
    "checkpoints": "checkpoints",
    "clip_vision": "clip_vision",
    "controlnet": "controlnet",
    "ipadapter": "ipadapter",
    "ultralytics": "ultralytics",
    "upscale_models": "upscale_models",
    "vae": "vae",
    "loras": "loras",
}

comfy_models_root = Path(COMFY_DIR) / "models"

def safe_symlink(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(src, dst)

linked_files = []
for ds_cat, comfy_cat in CATEGORY_MAP.items():
    src_dir = DATASET_ROOT / ds_cat
    if not src_dir.exists():
        continue
    dst_dir = comfy_models_root / comfy_cat
    dst_dir.mkdir(parents=True, exist_ok=True)

    for file_path in src_dir.rglob("*"):
        if file_path.is_file():
            rel_path = file_path.relative_to(src_dir)
            # 1. Preserving relative hierarchy
            safe_symlink(file_path, dst_dir / rel_path)
            # 2. Flattened filename at category root
            safe_symlink(file_path, dst_dir / file_path.name)
            # 3. For ultralytics, guarantee bbox/face_yolov8n.pt is present
            if ds_cat == "ultralytics" and "bbox" in str(rel_path):
                safe_symlink(file_path, dst_dir / "bbox" / file_path.name)
            linked_files.append((comfy_cat, file_path.name))

print("\n✅ Linked Models Inventory:")
for cat, name in sorted(linked_files):
    print(f"  • [{cat:<15}] {name}")

## 🚀 Cell 3: Start ComfyUI Background Server (Non-Blocking & Anti-Hang)
Redirects stdout/stderr directly to `/kaggle/working/comfyui_server.log` to prevent pipe deadlocks.

In [ ]:
import time
import urllib.request
import subprocess

LOG_FILE = os.path.join(WORKSPACE, "comfyui_server.log")
COMFY_URL = "http://127.0.0.1:8188"

def is_server_alive():
    try:
        with urllib.request.urlopen(f"{COMFY_URL}/system_stats", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False

if not is_server_alive():
    log_handle = open(LOG_FILE, "w", encoding="utf-8")
    server_proc = subprocess.Popen(
        [
            sys.executable, "main.py",
            "--listen", "0.0.0.0",
            "--port", "8188",
            "--fp16-vae",
            "--disable-xformers"
        ],
        cwd=COMFY_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        close_fds=True
    )
    print("⏳ Starting ComfyUI server in background...")
    for i in range(90):
        if is_server_alive():
            print(f"🚀 ComfyUI Server is ONLINE at {COMFY_URL} (took {i*2}s)")
            break
        time.sleep(2)
    else:
        print("❌ Server startup timed out. Last 40 lines of log:")
        with open(LOG_FILE, "r", encoding="utf-8") as lf:
            print("".join(lf.readlines()[-40:]))
        raise RuntimeError("ComfyUI failed to start.")
else:
    print(f"🚀 ComfyUI Server is already running at {COMFY_URL}")

## ⚙️ Cell 4: The "6 + 2" Specialized Workflow Engine
Implements all 8 production contracts bound directly to exact model filenames.

In [ ]:
import sys
sys.path.insert(0, WORKSPACE)
from kaggle_ai_engine.comfyui_production_runner import NovelVisualiserEngine
engine = NovelVisualiserEngine(server_url=COMFY_URL, workspace=WORKSPACE)
print("✅ Cell 4 Complete: NovelVisualiserEngine (8 Specialized Workflows) initialized.")

## 🎬 Cell 5: Run Full End-to-End Production Chain & Display Gallery
Generates Sunny & Nephis turnaround/expressions, Crimson Spire hall plate + depth, Midnight Shard dagger prop, solo shot, multi-subject regional shot, environment vista, surgical inpainting, and 4K upscale.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# 1. WF-FOUNDRY-CHAR
sunny_ref = engine.wf_foundry_char(
    char_id="sunny",
    identity_tags="1boy, pale skin, messy black hair, sharp dark onyx eyes, slender build, black leather tunic",
    mode="turnaround",
    seed=101
)
nephis_ref = engine.wf_foundry_char(
    char_id="nephis",
    identity_tags="1girl, silver hair, calm grey eyes, tall athletic build, silver knight armor, stoic expression",
    mode="expressions",
    seed=102
)

# 2. WF-FOUNDRY-LOC
crimson_spire_loc = engine.wf_foundry_loc(
    project_id="shadow_slave",
    arc_id="forgotten_shore",
    loc_id="crimson_spire_hall",
    landscape_tags="ancient ruined gothic cathedral interior, obsidian pillars, crimson twilight rays through broken stained glass",
    seed=202
)

# 3. WF-FOUNDRY-PROP
dagger_prop = engine.wf_foundry_prop(
    prop_id="midnight_shard",
    item_tags="curved silver dagger, obsidian hilt, glowing runic engravings, dark fantasy artifact",
    scale_cm=38.0,
    seed=303
)

# 4. WF-SHOT-SOLO
shot_solo = engine.wf_shot_solo(
    shot_id="ch01_s01",
    prompt_8block="medium shot, 1boy, pale skin, messy black hair, dark leather armor, inside ruined gothic cathedral",
    face_ref_path=str(sunny_ref),
    loc_depth_path=str(crimson_spire_loc["depth"]),
    seed=404
)

# 5. WF-SHOT-MULTI
shot_multi = engine.wf_shot_multi(
    shot_id="ch01_s02",
    bg_prompt="ancient ruined gothic cathedral interior, obsidian pillars, crimson twilight",
    subj1_prompt="1boy, pale skin, messy black hair, sharp dark onyx eyes, dark leather tunic, combat stance",
    subj2_prompt="1girl, silver hair, calm grey eyes, silver knight armor, composed stance",
    subj1_face_path=str(sunny_ref),
    subj2_face_path=str(nephis_ref),
    loc_depth_path=str(crimson_spire_loc["depth"]),
    seed=505
)

# 6. WF-SHOT-ENV
shot_env = engine.wf_shot_env(
    shot_id="ch01_s03",
    env_prompt="storm brewing over dark jagged mountains, giant crimson spire on horizon, ominous lightning",
    loc_depth_path=str(crimson_spire_loc["depth"]),
    seed=606
)

# 7. WF-EDIT-INPAINT
shot_repaired = engine.wf_edit_inpaint(
    shot_id="ch01_s01_fix",
    failed_image_path=str(shot_solo),
    repair_prompt="holding curved silver dagger with obsidian hilt, glowing runic engravings, gauntleted hand",
    defect_bbox=(680, 480, 320, 280),
    denoise=0.62,
    seed=707
)

# 8. WF-MASTER-UPSCALE
shot_mastered = engine.wf_master_upscale(
    shot_id="ch01_s01_master",
    approved_image_path=str(shot_repaired),
    prompt_tags="1boy, messy black hair, dark leather armor, holding silver dagger, gothic cathedral",
    scale_factor=2.0,
    denoise=0.24,
    seed=808
)

gallery = [
    ("01: WF-FOUNDRY-CHAR (Sunny)", sunny_ref),
    ("02: WF-FOUNDRY-LOC (Master)", crimson_spire_loc["master"]),
    ("02b: WF-FOUNDRY-LOC (Depth)", crimson_spire_loc["depth"]),
    ("03: WF-FOUNDRY-PROP", dagger_prop["image"]),
    ("04: WF-SHOT-SOLO", shot_solo),
    ("05: WF-SHOT-MULTI (Regional)", shot_multi),
    ("06: WF-SHOT-ENV", shot_env),
    ("07: WF-EDIT-INPAINT", shot_repaired),
    ("08: WF-MASTER-UPSCALE", shot_mastered),
]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, (title, img_p) in zip(axes.flatten(), gallery):
    ax.imshow(Image.open(img_p))
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.axis("off")
plt.tight_layout()
plt.show()